### This script measures risk adjusted performance of stocks of AAPL by calculating Sharpe ratio, Treynor ratio, and Sortino ratio. I will define the ratios and their significance as we calculate them.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import pandas as pd
import yfinance as yf
import os

#### Load historical data of stocks of AAPL starting from date (YYYY-MM-DD) 2021-01-04 to end date 2025-12-31 as a csv file.

In [2]:
stocks = {
    "^GSPC": "SP500.csv",
    "AAPL": "AAPL.csv",
    "MSFT": "MSFT.csv",
}
download_stocks = False
data={}

## Download AAPL data
if(download_stocks):
    for ticker, filename in stocks.items():
        print(ticker)
        data[ticker] = yf.download(ticker,start="2021-01-01",end="2026-01-01",auto_adjust=False)
        data[ticker].to_csv(filename)
else:
    aapl = pd.read_csv("AAPL.csv")
    sp500_data = pd.read_csv("SP500.csv")
    msft = pd.read_csv("MSFT.csv")
    
riskfree = pd.read_csv("DGS3MO.csv")

#### We need to clean data first. First two rows are irrelevant. Convert strings into numerics, and setting date of trading as an index of data.

In [3]:
## Set date as the index of the dataframe
def set_date_as_index(data,column):
    data[column] = pd.to_datetime(data[column])
    data = data.set_index(column)
    data.index.rename("Date", inplace=True)
    return data

#### Run this cell only once at the beginning.

In [4]:
#### Remove first 2 rows because actual data starts from row 2.
aapl = aapl.iloc[2:]
sp500_data = sp500_data.iloc[2:]

aapl = set_date_as_index(aapl,"Price")
sp500_data = set_date_as_index(sp500_data, "Price")
riskfree = set_date_as_index(riskfree, "observation_date")

## Convert string values of Adjusted closing price of a stock to numeric 
aapl["Adj Close"] = pd.to_numeric(aapl["Adj Close"])
sp500_data["Adj Close"] = pd.to_numeric(sp500_data["Adj Close"])

#### Calculate daily returns $$R_d = \frac{(P_t - P_{t-1})}{P_{t-1}},$$ where $P_t$ = Adjusted closing price on day 't', $P_{t-1}$ = Adjusted closing price on day 't-1'.

In [5]:
aapl["aapl_return"] = aapl["Adj Close"].pct_change()
# plt.plot(aapl["aapl_return"])
aapl.head()

,Adj Close,Close,High,Low,Open,Volume,aapl_return
Date,,,,,,,
2021-01-04,125.632484,129.41000366210938,133.61000061035156,126.76000213623047,133.52000427246094,143301900,NaN
2021-01-05,127.185799,131.00999450683594,131.74000549316406,128.42999267578125,128.88999938964844,97664900,0.012364
2021-01-06,122.904518,126.5999984741211,131.0500030517578,126.37999725341797,127.72000122070312,155088000,-0.033662
2021-01-07,127.098412,130.9199981689453,131.6300048828125,127.86000061035156,128.36000061035156,109578200,0.034123
2021-01-08,128.195480,132.0500030517578,132.6300048828125,130.22999572753906,132.42999267578125,105158200,0.008632


#### Load the 3-month U.S. Treasury bill annualized rate and convert it to a daily risk-free rate using compound interest over 252 trading days per year: $$rate_d = (1 + rate_A)^{1/252} - 1$$ where $rate_d$ is the daily risk-free rate, and $rate_A$ is the annualized risk-free rate.

In [6]:
riskfree["rf_daily"] = (1 + riskfree["DGS3MO"])**(1/252) - 1
print(riskfree.head())

            DGS3MO  rf_daily
Date                        
2021-01-04    0.09  0.000342
2021-01-05    0.09  0.000342
2021-01-06    0.09  0.000342
2021-01-07    0.09  0.000342
2021-01-08    0.08  0.000305


#### Merge the datasets using the AAPL trading dates as the reference, omitting U.S. Treasury bill dates that are not present in the AAPL stock data.

In [7]:
## Check for missing dates in aapl and risk-free return data
missing_dates_data = aapl.index.difference(riskfree.index)
print(missing_dates_data)
missing_dates_rf = riskfree.index.difference(aapl.index)
print(missing_dates_rf)

merge_data = pd.merge(aapl, riskfree, how='left', left_index=True, right_index=True) 

DatetimeIndex([], dtype='datetime64[ns]', name='Date', freq=None)
DatetimeIndex(['2021-01-18', '2021-02-15', '2021-04-02', '2021-05-31',
               '2021-07-05', '2021-09-06', '2021-11-25', '2021-12-24',
               '2022-01-17', '2022-02-21', '2022-04-15', '2022-05-30',
               '2022-06-20', '2022-07-04', '2022-09-05', '2022-11-24',
               '2022-12-26', '2023-01-02', '2023-01-16', '2023-02-20',
               '2023-04-07', '2023-05-29', '2023-06-19', '2023-07-04',
               '2023-09-04', '2023-11-23', '2023-12-25', '2024-01-01',
               '2024-01-15', '2024-02-19', '2024-03-29', '2024-05-27',
               '2024-06-19', '2024-07-04', '2024-09-02', '2024-11-28',
               '2024-12-25', '2025-01-01', '2025-01-09', '2025-01-20',
               '2025-02-17', '2025-04-18', '2025-05-26', '2025-06-19',
               '2025-07-04', '2025-09-01', '2025-11-27', '2025-12-25'],
              dtype='datetime64[ns]', name='Date', freq=None)


#### Calculate the excess return of AAPL as $$R_{\mathrm{excess}} = R_d - R_f,$$ where $R_d$ is the daily AAPL return and $R_f$ is the daily risk-free return. Also calculate the daily market return from the S&P 500 data.

In [8]:
merge_data["aapl_excess_return"] = merge_data["aapl_return"] - merge_data["rf_daily"]
sp500_data["market_return"] = sp500_data["Adj Close"].pct_change()

#### Prepare the final risk data containing all the relevant information we require to calculate risk adjusted performance.

In [9]:
risk_data = pd.concat([
    merge_data["aapl_return"],
    merge_data["aapl_excess_return"],
    riskfree["rf_daily"],
    sp500_data["market_return"]
], axis=1)

# risk_data.columns = ["aapl_return","aapl_excess_return", "rf_daily", "market_return"]
risk_data = risk_data.dropna()
risk_data.head()

,aapl_return,aapl_excess_return,rf_daily,market_return
Date,,,,
2021-01-05,0.012364,0.012022,0.000342,0.007083
2021-01-06,-0.033662,-0.034004,0.000342,0.005710
2021-01-07,0.034123,0.033781,0.000342,0.014847
2021-01-08,0.008632,0.008326,0.000305,0.005492
2021-01-11,-0.023249,-0.023555,0.000305,-0.006555


## Calculate the Sharpe ratio, 
$$ S = \frac{\langle R_d - R_f \rangle}{\sigma_d},$$
where $R_d$ is the daily return, $R_f$ is the daily risk-free return, and $\sigma_d$ is the standard deviation of the daily returns. The notation $\langle\cdot\rangle$ denotes the average over the full period for which the excess return is calculated. The Sharpe ratio measures **the asset's excess return per unit of total risk.**


In [10]:
Sharpe_ratio = risk_data["aapl_excess_return"].mean()/risk_data["aapl_excess_return"].std()
print("Sharpe ratio:",Sharpe_ratio)

Sharpe ratio: -0.2428724472705111


## Calculate the Treynor ratio,

$$
T = \frac{\langle R_d - R_f \rangle}{\beta},
$$

where

$$
\beta = \frac{\operatorname{Cov}(R_d,R_m)}
{\operatorname{Var}(R_m)}.
$$

Here, $\beta$ measures the sensitivity of the asset's daily returns $(R_d)$ to market returns $(R_m)$. A positive $\beta$ indicates that the asset generally moves in the same direction as the market, while a negative $\beta$ indicates that it tends to move in the opposite direction. A larger magnitude of $\beta$ indicates greater sensitivity to market movements.

The Treynor ratio therefore measures the **excess return generated per unit of systematic (market) risk**.

In [11]:
covariance = risk_data["aapl_return"].cov(risk_data["market_return"])
market_variance = risk_data["market_return"].var()

beta = covariance / market_variance

print(r"$\beta$:",beta)
Treynor_ratio = risk_data["aapl_excess_return"].mean()/beta
print(r'Treynor ratio:',Treynor_ratio)

$\beta$: 1.2456832813979604
Treynor ratio: -0.003473061237155221
